In [1]:
import AILibs
import numpy

dataset_root_path = "/users/michal/datasets/ECG5000/"

dataset_train   = AILibs.datasets.ECG5000Dataset(dataset_root_path, split='TRAIN')
dataset_test    = AILibs.datasets.ECG5000Dataset(dataset_root_path, split='TEST')

x_train = dataset_train.features
y_train = dataset_train.labels

x_test = dataset_test.features
y_test = dataset_test.labels

num_classes = dataset_train.num_classes

print("Extracting features")
features_extractor = AILibs.features.Catch22Features(x_train)
#features_extractor = AILibs.features.RocketFeatures(x_train, 32)

# obtain features for train and test sets
z_train = features_extractor.forward(x_train)
z_test  = features_extractor.forward(x_test)

#z_train = numpy.reshape(x_train, (x_train.shape[0], -1))
#z_test  = numpy.reshape(x_test, (x_test.shape[0], -1))

print("Feature shape: ", z_train.shape)
print("Test feature shape: ", z_test.shape)

print("Training forest")


# one hot encoding
y_one_hot = numpy.eye(num_classes)[y_train.astype(int)]


#forest = AILibs.forest.RandomForest() 
# forest.fit(z_train, y_one_hot, max_depth=6, num_trees=256, num_subsamples=-1, num_random_candidates=16)
  

forest = AILibs.forest.RandomBoostingForest()
forest.fit(z_train, y_one_hot, max_depth=6, num_trees=128)




Loading ECG5000 TRAIN split...
Loaded 500 samples.
Feature shape per sample: (140, 1) (seq_length, num_features)
Number of unique classes found: 5
Loading ECG5000 TEST split...
Loaded 4500 samples.
Feature shape per sample: (140, 1) (seq_length, num_features)
Number of unique classes found: 5
Extracting features
Feature shape:  (500, 22)
Test feature shape:  (4500, 22)
Training forest


In [2]:


print("Predicting with Random Forest...")
y_pred = forest.predict_batch(z_test)


metrics_test = AILibs.metrics.classification_evaluation(y_test, y_pred,num_classes)

for key, value in metrics_test.items():
    print(f"{key}: {value}")

    

Predicting with Random Forest...
n_samples: 4500
num_classes: 5
accuracy: 0.93444
macro_precision: 0.68543
macro_recall: 0.49963
macro_f1_score: 0.54226
macro_mcc: 0.5477
macro_specificity: 0.97635
macro_balanced_accuracy: 0.73799
macro_iou: 0.4655
macro_dice: 0.54226
tp_per_class: [2599, 1537, 25, 44, 0]
tn_per_class: [1773, 2734, 4407, 4313, 4478]
fp_per_class: [100, 176, 7, 12, 0]
fn_per_class: [28, 53, 61, 131, 22]
precision_per_class: [0.96295, 0.89726, 0.78125, 0.78571, 0.0]
recall_per_class: [0.98934, 0.96667, 0.2907, 0.25143, 0.0]
f1_score_per_class: [0.97597, 0.93067, 0.42373, 0.38095, 0.0]
